# 01 — Quickstart: Connect, Ping, and Run Your First Workflow

This notebook walks through the absolute basics of the **Multigen** multi-agent orchestration framework:

1. Install the SDK and import the client
2. Connect to the running orchestrator at `http://localhost:8009`
3. Ping the server to verify connectivity
4. List registered agents
5. Run a simple `EchoAgent` workflow using the `WorkflowBuilder` DSL
6. Inspect workflow state and metrics

> **Pre-requisite**: Make sure the Multigen orchestrator is running (`docker compose up` or direct `uvicorn`).

## 1. Install the SDK

If you are running this notebook outside the repo virtual environment, install the SDK from the local path:

In [ ]:
# Uncomment the line below if the SDK is not already installed in your environment
# !pip install -e ../sdk

## 2. Import the SDK

`SyncMultigenClient` is the notebook-friendly synchronous façade over the async client.
`WorkflowBuilder` provides a fluent DSL for constructing workflow definitions.

In [ ]:
import json
import time
import pprint

from multigen import SyncMultigenClient
from multigen.dsl import WorkflowBuilder

print("SDK imported successfully.")

## 3. Connect to the Orchestrator

Create a `SyncMultigenClient` pointing at the local orchestrator. The client manages its own event loop, so all calls are blocking (safe for notebooks).

In [ ]:
ORCHESTRATOR_URL = "http://localhost:8009"

client = SyncMultigenClient(base_url=ORCHESTRATOR_URL, timeout=30.0)
print(f"Client created → {ORCHESTRATOR_URL}")

## 4. Ping the Server

`ping()` sends a lightweight `GET /health` request and returns `True` if the server is reachable.

In [ ]:
is_alive = client.ping()
print(f"Orchestrator reachable: {is_alive}")
assert is_alive, "Orchestrator is not running — start it before proceeding."

## 5. List Registered Agents

`list_agents()` returns all agents currently registered with the orchestrator.

In [ ]:
agents = client.list_agents()
print(f"Found {len(agents)} registered agent(s):\n")
for a in agents:
    print(f"  • {a.get('name', a)}")

## 6. Build a Simple EchoAgent Workflow

The `WorkflowBuilder` DSL lets you construct a workflow as a chain of steps. Each `.step()` call adds a named step bound to an agent.

Here we build a single-step workflow that passes a greeting message through `EchoAgent`.

In [ ]:
dsl_payload = (
    WorkflowBuilder()
    .step(
        "echo",
        agent="EchoAgent",
        params={"message": "Hello from Multigen Quickstart!"},
    )
    .build(payload={"user": "notebook"})
)

print("DSL payload:")
print(json.dumps(dsl_payload, indent=2))

## 7. Submit the Workflow

`run_workflow()` posts the DSL to `POST /workflows/run` and immediately returns a `RunResponse` containing the `instance_id` of the newly launched workflow instance.

In [ ]:
response = client.run_workflow(
    dsl=dsl_payload["dsl"],
    payload=dsl_payload["payload"],
)

instance_id = response.instance_id
print(f"Workflow launched!  instance_id = {instance_id}")

## 8. Wait for Completion and Check State

`get_state()` retrieves all persisted node outputs from MongoDB. We poll until the state is non-empty, meaning at least one node has completed.

In [ ]:
def wait_for_completion(client, instance_id, timeout=30, poll_interval=1):
    """Poll get_state until at least one node has completed or timeout elapses."""
    start = time.time()
    while time.time() - start < timeout:
        state = client.get_state(instance_id)
        if state.nodes:
            return state
        print("  waiting...", end="\r")
        time.sleep(poll_interval)
    return client.get_state(instance_id)

state = wait_for_completion(client, instance_id)
print(f"\nWorkflow state — {state.count} node output(s) recorded:")
for ns in state.nodes:
    print(f"  node: {ns.node_id}")
    pprint.pprint(ns.output, indent=4)

## 9. Retrieve Workflow Metrics

`get_metrics()` returns execution counters: how many nodes ran, whether any reflections or fan-outs were triggered, and error counts.

In [ ]:
metrics = client.get_metrics(instance_id)

print("Workflow Metrics")
print("----------------")
print(f"  nodes_executed       : {metrics.nodes_executed}")
print(f"  nodes_skipped        : {metrics.nodes_skipped}")
print(f"  reflections_triggered: {metrics.reflections_triggered}")
print(f"  fan_outs_executed    : {metrics.fan_outs_executed}")
print(f"  circuit_breaker_trips: {metrics.circuit_breaker_trips}")
print(f"  error_count          : {metrics.error_count}")

## 10. Clean Up

Always close the client when finished to release the underlying HTTP connection pool and event loop.

In [ ]:
client.close()
print("Client closed. Quickstart complete!")

---

## Summary

| Step | Method | Purpose |
|------|--------|---------|
| Connect | `SyncMultigenClient(url)` | Create a synchronous client |
| Health | `client.ping()` | Verify the orchestrator is alive |
| Discovery | `client.list_agents()` | See what agents are available |
| Build DSL | `WorkflowBuilder().step(...).build()` | Construct a workflow definition |
| Launch | `client.run_workflow(dsl=..., payload=...)` | Submit and get `instance_id` |
| Inspect | `client.get_state(id)` | Read persisted node outputs |
| Metrics | `client.get_metrics(id)` | View execution counters |

**Next**: See `02_graph_workflow.ipynb` for building and running directed graph workflows.